<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/ctc_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#encoding=utf-8

#The code make the Full-mell Spectrogram feature and save it as ark and scp
#Author:  Richardfan
#Date:    2018.4.24

import torchaudio
import librosa
import numpy as np
import struct
import sys
import scipy.signal

class KaldiWriteOut(object):
    def __init__(self, ark_path, scp_path):
        self.ark_path = ark_path
        self.scp_path = scp_path
	self.ark_file_write = open(ark_path, 'wb')
        self.scp_file_write = open(scp_path, 'w')
        self.pos = 0

    def write_kaldi_mat(self, utt_id, utt_mat):
        utt_mat = np.asarray(utt_mat, dtype=np.float32)
        rows, cols = utt_mat.shape
        self.ark_file_write.write(struct.pack('<%ds'%(len(utt_id)), utt_id))
        self.ark_file_write.write(struct.pack('<cxcccc', ' ', 'B', 'F', 'M', ' '))
        self.ark_file_write.write(struct.pack('<bi', 4, rows))
        self.ark_file_write.write(struct.pack('<bi', 4, cols))
        self.ark_file_write.write(utt_mat)
        self.pos += len(utt_id) + 1
        self.scp_file_write.write(utt_id + ' ' + self.ark_path + ':' + str(self.pos) + '\n')
        self.pos += 3 * 5 + (rows * cols * 4)

    def close(self):
        self.ark_file_write.close()
        self.scp_file_write.close()

def load_audio(path):
    '''
    Input:
        path     : string 载入音频的路径
    Output:
        sound    : numpy.ndarray 单声道音频数据，如果是多声道进行平均
    '''
    sound, _ = torchaudio.load(path)
    sound = sound.numpy()
    if len(sound.shape) > 1:
        if sound.shape[1] == 1:
            sound = sound.squeeze()
        else:
            sound - sound.mean(axis=1)
    return sound

def parse_audio(path, audio_conf, windows, normalize=True):
    '''
    Input:
        path       : string 导入音频的路径
        audio_conf : dcit 求频谱的音频参数
        windows    : dict 加窗类型
    Output:
        spect      : ndarray  每帧的频谱
    '''
    y = load_audio(path)
    n_fft = int(audio_conf['sample_rate']*audio_conf["window_size"])
    win_length = n_fft
    hop_length = int(audio_conf['sample_rate']*audio_conf['window_stride'])
    window = windows[audio_conf['window']]
    D = librosa.stft(y, n_fft=n_fft, hop_length=hop_length,
                        win_length=win_length, window=window)
    spect, phase = librosa.magphase(D)

    spect = np.log1p(spect)

    if normalize:
        mean = spect.mean()
        std = spect.std()
        spect = np.add(spect, -mean)
        spect = np.divide(spect, std)

    return spect.transpose()

def make_spectrum(wave_path, ark_file, scp_file):
    windows = {'hamming':scipy.signal.hamming, 'hann':scipy.signal.hann, 'blackman':scipy.signal.blackman,
                'bartlett':scipy.signal.bartlett}
    audio_conf = {"sample_rate":16000, 'window_size':0.025, 'window_stride':0.01, 'window': 'hamming'}
    arkwriter = KaldiWriteOut(ark_file, scp_file)
    with open(wave_path, 'r') as rf:
        i = 0
        for lines in rf.readlines():
            utt_id, path = lines.strip().split()
            utt_mat = parse_audio(path, audio_conf, windows, normalize=True)
            arkwriter.write_kaldi_mat(utt_id, utt_mat)
            i += 1
            if i %10 == 0:
                print("Processed %d sentences..." % i)
        arkwriter.close()
        print("Done. Processed %d sentences..." % i)

if __name__ == '__main__':
    if len(sys.argv) != 4:
        print("Usage: python "+sys.argv[0] + ' [wav_path] [ark file to write] [scp file to write]')
        sys.exit(1)
    wave_path = sys.argv[1]
    ark_file = sys.argv[2]
    scp_file = sys.argv[3]
    make_spectrum(wave_path, ark_file, scp_file)


In [ ]:
#encoding=utf-8

import os
import sys
import argparse

parser = argparse.ArgumentParser(description="Normalize the phoneme on TIMIT")
parser.add_argument("--map", default="./decode_map_48-39/phones.60-48-39.map", help="The map file")
parser.add_argument("--to", default=48, help="Determine how many phonemes to map")
parser.add_argument("--src", default='./data_prepare/train/phn_text', help="The source file to mapping")
parser.add_argument("--tgt", default='./data_prepare/train/48_text' ,help="The target file after mapping")

def main():
    args = parser.parse_args()
    if not os.path.exists(args.map) or not os.path.exists(args.src):
        print("Map file or source file not exist !")
        sys.exit(1)

    map_dict = {}
    with open(args.map) as f:
        for line in f.readlines():
            line = line.strip().split('\t')
            if args.to == "60-48":
                if len(line) == 1:
                    map_dict[line[0]] = ""
                else:
                    map_dict[line[0]] = line[1]
            elif args.to == "60-39":
                if len(line) == 1:
                    map_dict[line[0]] = ""
                else:
                    map_dict[line[0]] = line[2]
            elif args.to == "48-39":
                if len(line) == 3:
                    map_dict[line[1]] = line[2]
            else:
                print("%s phonemes are not supported" % args.to)
                sys.exit(1)

    with open(args.src, 'r') as rf, open(args.tgt, 'w') as wf:
        for line in rf.readlines():
            line = line.strip().split(' ')
            uttid, utt = line[0], line[1:]
            map_utt = [ map_dict[phone] for phone in utt if map_dict[phone] != "" ]
            wf.writelines(uttid + ' ' + ' '.join(map_utt) + '\n')

if __name__ == "__main__":
    main()

In [ ]:
#!/bin/bash

#此文件用来得到训练集，验证集和测试集的音频路径文件和转录文本即标签文件以便后续处理
#输入的参数时TIMIT数据库的路径。
#更换数据集之后，因为数据的目录结构不一致，需要对此脚本进行简单的修改。

if [ $# -ne 2 ]; then
   echo "Need directory of TIMIT dataset !"
   exit 1;
fi

conf_dir=`pwd`/conf
prepare_dir=`pwd`/data
map_file=$conf_dir/phones.60-48-39.map
phoneme_map=$2

. path.sh
sph2pipe=$KALDI_ROOT/tools/sph2pipe_v2.5/sph2pipe
if [ ! -x $sph2pipe ]; then
   echo "Could not find (or execute) the sph2pipe program at $sph2pipe";
   exit 1;
fi

[ -f $conf_dir/test_spk.list ] || error_exit "$PROG: Eval-set speaker list not found.";
[ -f $conf_dir/dev_spk.list ] || error_exit "$PROG: dev-set speaker list not found.";

#根据数据库train，test的名称修改，有时候下载下来train可能是大写或者是其他形式
train_dir=train
test_dir=test

ls -d "$1"/$train_dir/dr*/* | sed -e "s:^.*/::" > $conf_dir/train_spk.list

tmpdir=`pwd`/tmp
mkdir -p $tmpdir $prepare_dir
for x in train dev test; do
  if [ ! -d $prepare_dir/$x ]; then
      mkdir -p $prepare_dir/$x
  fi

  # 只使用 si & sx 的语音.
  find $1/{$train_dir,$test_dir} -not \( -iname 'SA*' \) -iname '*.WAV' \
    | grep -f $conf_dir/${x}_spk.list > $tmpdir/${x}_sph.flist

  #获得每句话的id标识
  sed -e 's:.*/\(.*\)/\(.*\).WAV$:\1_\2:i' $tmpdir/${x}_sph.flist \
    > $tmpdir/${x}_sph.uttids

  #生成wav.scp,即每句话的音频路径
  paste -d" " $tmpdir/${x}_sph.uttids $tmpdir/${x}_sph.flist \
    | sort -k1,1 > $prepare_dir/$x/wav.scp

  awk '{printf("%s '$sph2pipe' -f wav %s |\n", $1, $2);}' < $prepare_dir/$x/wav.scp > $prepare_dir/$x/wav_sph.scp

  for y in wrd phn; do
    find $1/{$train_dir,$test_dir} -not \( -iname 'SA*' \) -iname '*.'$y'' \
        | grep -f $conf_dir/${x}_spk.list > $tmpdir/${x}_txt.flist
    sed -e 's:.*/\(.*\)/\(.*\).'$y'$:\1_\2:i' $tmpdir/${x}_txt.flist \
        > $tmpdir/${x}_txt.uttids
    while read line; do
        [ -f $line ] || error_exit "Cannot find transcription file '$line'";
        cut -f3 -d' ' "$line" | tr '\n' ' ' | sed -e 's: *$:\n:'
    done < $tmpdir/${x}_txt.flist > $tmpdir/${x}_txt.trans

    #将句子标识（uttid）和文本标签放在一行并按照uttid进行排序使其与音频路径顺序一致
    paste -d" " $tmpdir/${x}_txt.uttids $tmpdir/${x}_txt.trans \
        | sort -k1,1 > $tmpdir/${x}.trans

    #生成文本标签
    cat $tmpdir/${x}.trans | sort > $prepare_dir/$x/${y}_text || exit 1;
    if [ $y == phn ]; then
        cp $prepare_dir/$x/${y}_text $prepare_dir/$x/${y}_text.tmp
        python local/normalize_phone.py --map $map_file --to $phoneme_map --src $prepare_dir/$x/${y}_text.tmp --tgt $prepare_dir/$x/${y}_text
        rm -f $prepare_dir/$x/${y}_text.tmp
    fi
  done
done

rm -rf $tmpdir

echo "Data preparation succeeded"

#beam search.py

In [ ]:
#!/usr/bin/python
#encoding=utf-8

import math

LOG_ZERO = -99999999.0
LOG_ONE = 0.0

class BeamEntry:
    "information about one single beam at specific time-step"
    def __init__(self):
        self.prTotal=LOG_ZERO      # blank and non-blank
        self.prNonBlank=LOG_ZERO   # non-blank
        self.prBlank=LOG_ZERO      # blank
        self.y=()                  # labelling at current time-step


class BeamState:
    "information about beams at specific time-step"
    def __init__(self):
        self.entries={}

    def norm(self):
        "length-normalise probabilities to avoid penalising long labellings"
        for (k,v) in self.entries.items():
            labellingLen=len(self.entries[k].y)
            self.entries[k].prTotal=self.entries[k].prTotal*(1.0/(labellingLen if labellingLen else 1))

    def sort(self):
        "return beams sorted by probability"
        u=[v for (k,v) in self.entries.items()]
        s=sorted(u, reverse=True, key=lambda x:x.prTotal)
        return [x.y for x in s]

class ctcBeamSearch(object):
    def __init__(self, classes, beam_width, lm, lm_alpha=0.01, blank_index=0):
        self.classes = classes #인덱스를 문자로 바꿀 리스트
        self.beamWidth = beam_width #후보 수
        self.lm_alpha = lm_alpha #lm 가중치
        self.lm = lm #lm모델 이름
        self.blank_index = blank_index #ctc blank 토큰 인덱스

    def log_add_prob(self, log_x, log_y): #0 대신 로그 추가
        if log_x <= LOG_ZERO:
            return log_y
        if log_y <= LOG_ZERO:
            return log_x
        if (log_y - log_x) > 0.0:
            log_y, log_x = log_x, log_y
        return log_x + math.log(1 + math.exp(log_y - log_x))

    def calcExtPr(self, k, y, t, mat, beamState): #
        "probability for extending labelling y to y+k"

        # language model (char bigrams)
        bigramProb=LOG_ONE
        if self.lm: #
            c1=self.classes[y[-1]] if len(y) else ''
            c2=self.classes[k]
            bigramProb = self.lm.get_bi_prob(c1,c2) * self.lm_alpha

        # optical model (RNN)
        if len(y) and y[-1]==k and mat[t-1, self.blank_index] < 0.9:
            return math.log(mat[t, k]) + bigramProb + beamState.entries[y].prBlank
        else:
            return math.log(mat[t, k]) + bigramProb + beamState.entries[y].prTotal

    def addLabelling(self, beamState, y): #라벨 추가
        "adds labelling if it does not exist yet"
        if y not in beamState.entries:
            beamState.entries[y]=BeamEntry()

    def decode(self, inputs, inputs_list):
        '''
        mat : FloatTesnor batch * timesteps * class
        '''
        batches, maxT, maxC = inputs.size()
        res = []

        for batch in range(batches): #배치별 반복
            mat = inputs[batch].numpy()
            # Initialise beam state
            last=BeamState() #빔 상태 초기화
            y=()
            last.entries[y]=BeamEntry() #현재 위치에 어떤걸 넣을지 초기화
            last.entries[y].prBlank=LOG_ONE
            last.entries[y].prTotal=LOG_ONE

            # go over all time-steps
            for t in range(inputs_list[batch]): #타임 스텝
                curr=BeamState()
                #跳过概率很接近1的blank帧，增加解码速度
                if (1 - mat[t, self.blank_index]) < 0.1:
                    continue
                #取前beam个最好的结果
                BHat=last.sort()[0:self.beamWidth]
                # go over best labellings
                for y in BHat:
                    prNonBlank=LOG_ZERO
                    # if nonempty labelling
                    if len(y)>0:
                        #相同的y两种可能，加入重复或者加入空白,如果之前没有字符，在NonBlank概率为0
                        prNonBlank=last.entries[y].prNonBlank + math.log(mat[t, y[-1]])

                    # calc probabilities
                    prBlank = (last.entries[y].prTotal) + math.log(mat[t, self.blank_index])
                    # save result
                    self.addLabelling(curr, y)
                    curr.entries[y].y=y
                    curr.entries[y].prNonBlank = self.log_add_prob(curr.entries[y].prNonBlank, prNonBlank)
                    curr.entries[y].prBlank = self.log_add_prob(curr.entries[y].prBlank, prBlank)
                    prTotal = self.log_add_prob(prBlank, prNonBlank)
                    curr.entries[y].prTotal = self.log_add_prob(curr.entries[y].prTotal, prTotal)

                    #t时刻加入其它的label,此时Blank的概率为0，如果加入的label与最后一个相同，因为不能重复，所以上一个字符一定是blank
                    for k in range(maxC):
                        if k != self.blank_index:
                            newY=y+(k,)
                            prNonBlank=self.calcExtPr(k, y, t, mat, last)

                            # save result
                            self.addLabelling(curr, newY)
                            curr.entries[newY].y=newY
                            curr.entries[newY].prNonBlank = self.log_add_prob(curr.entries[newY].prNonBlank, prNonBlank)
                            curr.entries[newY].prTotal = self.log_add_prob(curr.entries[newY].prTotal, prNonBlank)

                # set new beam state
                last=curr

            BHat=last.sort()[0:self.beamWidth]
            # go over best labellings
            curr = BeamState()
            for y in BHat:
                newY = y
                c1 = self.classes[y[-1]]
                c2 = ""
                prNonBlank = last.entries[newY].prTotal + self.lm.get_bi_prob(c1, c2) * self.lm_alpha
                self.addLabelling(curr, newY)
                curr.entries[newY].y=newY
                curr.entries[newY].prNonBlank = self.log_add_prob(curr.entries[newY].prNonBlank, prNonBlank)
                curr.entries[newY].prTotal = self.log_add_prob(curr.entries[newY].prTotal, prNonBlank)

            last = curr
            # normalise probabilities according to labelling length
            last.norm()

            # sort by probability
            bestLabelling=last.sort()[0] # get most probable labelling

            # map labels to chars
            res_b =' '.join([self.classes[l] for l in bestLabelling])
            res.append(res_b)
        return res

In [ ]:
#!/usrbin/python
#encoding=utf-8

# Get n-gram propability from arpa file;

import re
import math

n_grams = ["unigram", 'bigram', 'trigram', '4gram', '5gram']

class LanguageModel:
    """
    New version of LanguageModel which can read the text arpa file ,which
    is generate from kennlm
    """
    def __init__(self, arpa_file=None, n_gram=2, start='<s>', end='</s>', unk='<unk>'):
        "Load arpa file to get words and prob"
        self.n_gram = n_gram
        self.start = start
        self.end = end
        self.unk = unk
        self.scale = math.log(10)    #arpa格式是以10为底的对数概率，转化为以e为底
        self.initngrams(arpa_file)

    def initngrams(self, fn):
        "internal init of word bigrams"
        self.unigram = {}
        self.bigram = {}
        if self.n_gram == 3:
            self.trigrame = {}

	    # go through text and create each bigrams
        f = open(fn, 'r')
        recording = 0
        for lines in f.readlines():
            line = lines.strip('\n')
            #a = re.match('gram', line)
            if line == "\\1-grams:":
                recording = 1
                continue
            if line == "\\2-grams:":
                recording = 2
                continue
            if recording == 1:
                line = line.split('\t')
                if len(line) == 3:
                    self.unigram[line[1]] = [self.scale * float(line[0]), self.scale * float(line[2])]   #save the prob and backoff prob
                elif len(line) == 2:
                    self.unigram[line[1]] = [self.scale * float(line[0]), 0.0]
            if recording == 2:
                line = line.split('\t')
                if len(line) == 3:
                    #print(line[1])
                    self.bigram[line[1]] = [self.scale * float(line[0]), self.scale * float(line[2])]
                elif len(line) == 2:
                    self.bigram[line[1]] = [self.scale * float(line[0]), 0.0]
        f.close()
        self.unigram['UNK'] = self.unigram[self.unk]


    def get_uni_prob(self, wid):
        "Returns unigram probabiliy of word"
        return self.unigram[wid][0]

    def get_bi_prob(self, w1, w2):
        '''
        Return bigrams probability p(w2 | w1)
        if bigrame does not exist, use backoff prob
        '''
        if w1 == '':
            w1 = self.start
        if w2 == '':
            w2 = self.end
        key = w1 + ' ' + w2
        if key not in self.bigram:
            return self.unigram[w1][1] + self.unigram[w2][0]
        else:
            return self.bigram[key][0]

    def score_bg(self, sentence):
        '''
        Score a sentence using bigram, return P(sentence)
        '''
        val = 0.0
        words = sentence.strip().split()
        val += self.get_bi_prob(self.start, words[0])
        for i in range(len(words)-1):
            val += self.get_bi_prob(words[i], words[i+1])
        val += self.get_bi_prob(words[-1], self.end)
        return val

if __name__ == "__main__":
    lm = LanguageModel('./data_prepare/bigram.arpa')
    #print(lm.bigram['你 好'])
    print(lm.get_bi_prob('', 'sil'))
    #print(lm.score_bg("中国 呼吸"))

In [ ]:
#/usr/bin/python
#encoding=utf-8

#greedy decoder and beamsearch decoder for ctc

import torch
import numpy as np

class Decoder(object):
    "解码器基类定义，作用是将模型的输出转化为文本使其能够与标签计算正确率"
    def __init__(self, int2char, space_idx = 1, blank_index = 0):
        '''
        int2char     :     将类别转化为字符标签
        space_idx    :     空格符号的索引，如果为为-1，表示空格不是一个类别
        blank_index  :     空白类的索引，默认设置为0
        '''
        self.int_to_char = int2char
        self.space_idx = space_idx
        self.blank_index = blank_index
        self.num_word = 0
        self.num_char = 0

    def decode(self):
        "解码函数，在GreedyDecoder和BeamDecoder继承类中实现"
        raise NotImplementedError;

    def phone_word_error(self, prob_tensor, frame_seq_len, targets, target_sizes):
        '''计算词错率和字符错误率
        Args:
            prob_tensor     :   模型的输出
            frame_seq_len   :   每个样本的帧长
            targets         :   样本标签
            target_sizes    :   每个样本标签的长度
        Returns:
            wer             :   词错率，以space为间隔分开作为词
            cer             :   字符错误率
        '''
        strings = self.decode(prob_tensor, frame_seq_len)
        targets = self._unflatten_targets(targets, target_sizes)
        target_strings = self._process_strings(self._convert_to_strings(targets))

        cer = 0
        wer = 0
        for x in range(len(target_strings)):
            cer += self.cer(strings[x], target_strings[x])
            wer += self.wer(strings[x], target_strings[x])
            self.num_word += len(target_strings[x].split())
            self.num_char += len(target_strings[x])
        return cer, wer

    def _unflatten_targets(self, targets, target_sizes):
        '''将标签按照每个样本的标签长度进行分割
        Args:
            targets        :    数字表示的标签
            target_sizes   :    每个样本标签的长度
        Returns:
            split_targets  :    得到的分割后的标签
        '''
        split_targets = []
        offset = 0
        for size in target_sizes:
            split_targets.append(targets[offset : offset + size])
            offset += size
        return split_targets

    def _process_strings(self, seqs, remove_rep = False):
        '''处理转化后的字符序列，包括去重复等，将list转化为string
        Args:
            seqs       :   待处理序列
            remove_rep :   是否去重复
        Returns:
            processed_strings  :  处理后的字符序列
        '''
        processed_strings = []
        for seq in seqs:
            string = self._process_string(seq, remove_rep)
            processed_strings.append(string)
        return processed_strings

    def _process_string(self, seq, remove_rep = False):
        string = ''
        for i, char in enumerate(seq):
            if char != self.int_to_char[self.blank_index]:
                if remove_rep and i != 0 and char == seq[i - 1]: #remove dumplicates
                    pass
                elif self.space_idx == -1:
                    string = string + ' '+ char
                elif char == self.int_to_char[self.space_idx]:
                    string += ' '
                else:
                    string = string + char
        return string

    def _convert_to_strings(self, seq, sizes=None):
        '''将数字序列的输出转化为字符序列
        Args:
            seqs       :   待转化序列
            sizes      :   每个样本序列的长度
        Returns:
            strings  :  转化后的字符序列
        '''
        strings = []
        for x in range(len(seq)):
            seq_len = sizes[x] if sizes is not None else len(seq[x])
            string = self._convert_to_string(seq[x], seq_len)
            strings.append(string)
        return strings

    def _convert_to_string(self, seq, sizes):
        result = []
        for i in range(sizes):
            result.append(self.int_to_char[seq[i]])
        if self.space_idx == -1:
            return result
        else:
            return ''.join(result)

    def wer(self, s1, s2):
        "将空格作为分割计算词错误率"
        b = set(s1.split() + s2.split())
        word2int = dict(zip(b, range(len(b))))

        w1 = [word2int[w] for w in s1.split()]
        w2 = [word2int[w] for w in s2.split()]
        return self._edit_distance(w1, w2)

    def cer(self, s1, s2):
        "计算字符错误率"
        return self._edit_distance(s1, s2)

    def _edit_distance(self, src_seq, tgt_seq):
        "计算两个序列的编辑距离，用来计算字符错误率"
        L1, L2 = len(src_seq), len(tgt_seq)
        if L1 == 0: return L2
        if L2 == 0: return L1
        # construct matrix of size (L1 + 1, L2 + 1)
        dist = [[0] * (L2 + 1) for i in range(L1 + 1)]
        for i in range(1, L2 + 1):
            dist[0][i] = dist[0][i-1] + 1
        for i in range(1, L1 + 1):
            dist[i][0] = dist[i-1][0] + 1
        for i in range(1, L1 + 1):
            for j in range(1, L2 + 1):
                if src_seq[i - 1] == tgt_seq[j - 1]:
                    cost = 0
                else:
                    cost = 1
                dist[i][j] = min(dist[i][j-1] + 1, dist[i-1][j] + 1, dist[i-1][j-1] + cost)
        return dist[L1][L2]


class GreedyDecoder(Decoder):
    "直接解码，把每一帧的输出概率最大的值作为输出值，而不是整个序列概率最大的值"
    def decode(self, prob_tensor, frame_seq_len):
        '''解码函数
        Args:
            prob_tensor   :   网络模型输出
            frame_seq_len :   每一样本的帧数
        Returns:
            解码得到的string，即识别结果
        '''
        prob_tensor = prob_tensor.transpose(0,1)
        _, decoded = torch.max(prob_tensor, 2)
        decoded = decoded.view(decoded.size(0), decoded.size(1))
        decoded = self._convert_to_strings(decoded.numpy(), frame_seq_len)
        return self._process_strings(decoded, remove_rep=True)

class BeamDecoder(Decoder):
    "Beam search 解码。解码结果为整个序列概率的最大值"
    def __init__(self, int2char, beam_width = 200, blank_index = 0, space_idx = -1, lm_path=None, lm_alpha=0.01):
        self.beam_width = beam_width
        super(BeamDecoder, self).__init__(int2char, space_idx=space_idx, blank_index=blank_index)

        import sys
        sys.path.append('../')
        import utils.BeamSearch as uBeam
        import utils.NgramLM as uNgram
        lm = uNgram.LanguageModel(arpa_file=lm_path)
        self._decoder = uBeam.ctcBeamSearch(int2char, beam_width, lm, lm_alpha=lm_alpha, blank_index = blank_index)

    def decode(self, prob_tensor, frame_seq_len=None):
        '''解码函数
        Args:
            prob_tensor   :   网络模型输出
            frame_seq_len :   每一样本的帧数
        Returns:
            res           :   解码得到的string，即识别结果
        '''
        probs = prob_tensor.transpose(0, 1)
        probs = torch.exp(probs)
        res = self._decoder.decode(probs, frame_seq_len)
        return res


if __name__ == '__main__':
    decoder = Decoder('abcde', 1, 2)
    print(decoder._convert_to_strings([[1,2,1,0,3],[1,2,1,1,1]]))

In [ ]:
#!/usr/bin/python
#encoding=utf-8

import torch
import kaldiio
import numpy as np
from torch.utils.data import Dataset, DataLoader

from utils.tools import load_wave, F_Mel, make_context, skip_feat

audio_conf = {"sample_rate":16000, 'window_size':0.025, 'window_stride':0.01, 'window': 'hamming'}

class Vocab(object):
    def __init__(self, vocab_file):
        self.vocab_file = vocab_file
        self.word2index = {"blank": 0, "UNK": 1}
        self.index2word = {0: "blank", 1: "UNK"}
        self.word2count = {}
        self.n_words = 2
        self.read_lang()

    def add_sentence(self, sentence):
        for word in sentence.split(' '):
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

    def read_lang(self):
        print("Reading vocabulary from {}".format(self.vocab_file))
        with open(self.vocab_file, 'r') as rf:
            line = rf.readline()
            while line:
                line = line.strip().split(' ')
                if len(line) > 1:
                    sen = ' '.join(line[1:])
                else:
                    sen = line[0]
                self.add_sentence(sen)
                line = rf.readline()
        print("Vocabulary size is {}".format(self.n_words))


class SpeechDataset(Dataset):
    def __init__(self, vocab, scp_path, lab_path, opts):
        self.vocab = vocab
        self.scp_path = scp_path
        self.lab_path = lab_path
        self.left_ctx = opts.left_ctx
        self.right_ctx = opts.right_ctx
        self.n_skip_frame = opts.n_skip_frame
        self.n_downsample = opts.n_downsample
        self.feature_type = opts.feature_type
        self.mel = opts.mel

        if opts.feature_type == "waveform":
            self.label_dict = process_label_file(label_file, self.out_type, self.class2int)
            self.item = []
            with open(wav_path, 'r') as f:
                for line in f.readlines():
                    utt, path = line.strip().split('\t')
                    self.item.append((path, self.label_dict[utt]))
        else:
            self.process_feature_label()

    def process_feature_label(self):
        path_dict = []
        #read the ark path
        with open(self.scp_path, 'r') as rf:
            line = rf.readline()
            while line:
                utt, path = line.strip().split(' ')
                path_dict.append((utt, path))
                line = rf.readline()

       	#read the label
        label_dict = dict()
        with open(self.lab_path, 'r') as rf:
            line = rf.readline()
            while line:
                utt, label = line.strip().split(' ', 1)
                label_dict[utt] = [self.vocab.word2index[c] if c in self.vocab.word2index else self.vocab.word2index['UNK'] for c in label.split()]
                line = rf.readline()

        assert len(path_dict) == len(label_dict)
        print("Reading %d lines from %s" % (len(label_dict), self.lab_path))

        self.item = []
        for i in range(len(path_dict)):
            utt, path = path_dict[i]
            self.item.append((path, label_dict[utt], utt))

    def __getitem__(self, idx):
        if self.feature_type == "waveform":
            path, label = self.item[idx]
            return (load_wave(path), label)
        else:
            path, label, utt = self.item[idx]
            feat = kaldiio.load_mat(path)
            feat= skip_feat(make_context(feat, self.left_ctx, self.right_ctx), self.n_skip_frame)
            seq_len, dim = feat.shape
            if seq_len % self.n_downsample != 0:
                pad_len = self.n_downsample - seq_len % self.n_downsample
                feat = np.vstack([feat, np.zeros((pad_len, dim))])
            if self.mel:
                return (F_Mel(torch.from_numpy(feat), audio_conf), label)
            else:
                return (torch.from_numpy(feat), torch.LongTensor(label), utt)

    def __len__(self):
        return len(self.item)

def create_input(batch):
    inputs_max_length = max(x[0].size(0) for x in batch)
    feat_size = batch[0][0].size(1)
    targets_max_length = max(x[1].size(0) for x in batch)
    batch_size = len(batch)
    batch_data = torch.zeros(batch_size, inputs_max_length, feat_size)
    batch_label = torch.zeros(batch_size, targets_max_length)
    input_sizes = torch.zeros(batch_size)
    target_sizes = torch.zeros(batch_size)
    utt_list = []

    for x in range(batch_size):
        feature, label, utt = batch[x]
        feature_length = feature.size(0)
        label_length = label.size(0)

        batch_data[x].narrow(0, 0, feature_length).copy_(feature)
        batch_label[x].narrow(0, 0, label_length).copy_(label)
        input_sizes[x] = feature_length / inputs_max_length
        target_sizes[x] = label_length
        utt_list.append(utt)
    return batch_data.float(), input_sizes.float(), batch_label.long(), target_sizes.long(), utt_list

'''
class torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False, sampler=None, batch_sampler=None, num_workers=0,
                                                    collate_fn=<function default_collate>, pin_memory=False, drop_last=False)
subclass of DataLoader and rewrite the collate_fn to form batch
'''

class SpeechDataLoader(DataLoader):
    def __init__(self, *args, **kwargs):
        super(SpeechDataLoader, self).__init__(*args, **kwargs)
        self.collate_fn = create_input

if __name__ == '__main__':
    dev_dataset = SpeechDataset()
    dev_dataloader = SpeechDataLoader(dev_dataset, batch_size=2, shuffle=True)

    import visdom
    viz = visdom.Visdom(env='fan')
    for i in range(1):
        show = dev_dataset[i][0].transpose(0, 1)
        text = dev_dataset[i][1]
        for num in range(len(text)):
            text[num] = dev_dataset.int2class[text[num]]
        text = ' '.join(text)
        opts = dict(title=text, xlabel='frame', ylabel='spectrum')
        viz.heatmap(show, opts = opts)

In [ ]:
#!/usr/bin/python
#encoding=utf-8

__author__ = 'Ruchao Fan'

import math
import torch
import numpy as np
#import librosa
#import torchaudio

def load_audio(path):
    """
    Args:
        path     : string 载入音频的路径
    Returns:
        sound    : numpy.ndarray 单声道音频数据，如果是多声道进行平均
    """
    sound, _ = torchaudio.load(path)
    sound = sound.numpy()
    if len(sound.shape) > 1:
        if sound.shape[1] == 1:
            sound = sound.squeeze()
        else:
            sound = sound.mean(axis=1)
    return sound

def load_wave(path, normalize=True):
    """
    Args:
        path     : string 载入音频的路径
    Returns:
    """
    sound = load_audio(path)
    wave = torch.FloatTensor(sound)
    if normalize:
        mean = wave.mean()
        std = wave.std()
        wave.add_(-mean)
        wave.div_(std)
    return wave

def F_Mel(fre_f, audio_conf):
    '''
    Input:
        fre_f       : FloatTensor log spectrum
        audio_conf  : 主要需要用到采样率
    Output:
        mel_f       : FloatTensor  换成mel频谱
    '''
    n_mels = fre_f.size(1)
    mel_bin = librosa.mel_frequencies(n_mels=n_mels, fmin=0, fmax=audio_conf["sample_rate"]/2) * audio_conf["window_size"]
    count = 0
    fre_f = fre_f.numpy().tolist()
    mel_f = []
    for frame in fre_f:
        mel_f_frame = []
        for i in range(n_mels):
            left = int(math.floor(mel_bin[i]))
            right = left + 1
            tmp = (frame[right] - frame[left]) * (mel_bin[i] - left) + frame[left]      #线性插值
            mel_f_frame.append(tmp)
        mel_f.append(mel_f_frame)
    return torch.FloatTensor(mel_f)

def make_context(feature, left, right):
    if left==0 and right == 0:
        return feature
    feature = [feature]
    for i in range(left):
        feature.append(np.vstack((feature[-1][0], feature[-1][:-1])))
    feature.reverse()
    for i in range(right):
        feature.append(np.vstack((feature[-1][1:], feature[-1][-1])))
    return np.hstack(feature)

def skip_feat(feature, skip):
    '''
    '''
    if skip == 1 or skip == 0:
        return feature
    skip_feature=[]
    for i in range(feature.shape[0]):
        if i % skip == 0:
            skip_feature.append(feature[i])
    return np.vstack(skip_feature)

def process_label_file(label_file, label_type, class2int):
    '''
    Input:
        label_file  : string  标签文件路径
        label_type  : string  标签类型(目前只支持字符和音素)
        class2int   : dict    标签和数字的对应关系
    Output:
        label_dict  : dict    所有句子的标签，每个句子是numpy类型
    '''
    label_dict = dict()
    f = open(label_file, 'r')
    for label in f.readlines():
        label = label.strip()
        label_list = []
        if label_type == 'char':
            utt = label.split('\t', 1)[0]
            label = label.split('\t', 1)[1]
            for i in range(len(label)):
                if label[i].lower() in class2int:
                    label_list.append(class2int[label[i].lower()])
                if label[i] == ' ':
                    label_list.append(class2int['SPACE'])
        else:
            label = label.split()
            utt = label[0]
            for i in range(1,len(label)):
                label_list.append(class2int[label[i]])
        label_dict[utt] = label_list
    f.close()
    return label_dict

'''
if __name__ == '__main__':
    import scipy.signal
    windows = {'hamming':scipy.signal.hamming, 'hann':scipy.signal.hann, 'blackman':scipy.signal.blackman,
            'bartlett':scipy.signal.bartlett}
    audio_conf = {"sample_rate":16000, 'window_size':0.025, 'window_stride':0.01, 'window': 'hamming'}
    path = '/home/fan/Audio_data/TIMIT/test/dr7/fdhc0/si1559.wav'
    spect = parse_audio(path, audio_conf, windows, normalize=True)
    mel_f = F_Mel(spect, audio_conf)
    wave = load_wav(path)
    print(wave)

    import visdom
    viz = visdom.Visdom(env='fan')
    viz.heatmap(spect.transpose(0, 1), opts=dict(title="Log Spectrum", xlabel="She had your dark suit in greasy wash water all year.", ylabel="Frequency"))
    viz.heatmap(mel_f.transpose(0, 1), opts=dict(title="Log Mel Spectrum", xlabel="She had your dark suit in greasy wash water all year.", ylabel="Frequency"))
    viz.line(wave.numpy())
'''

In [ ]:

import sys

if len(sys.argv) != 2:
    print("We need training text to generate the modelling units.")
    sys.exit(1)

train_text = sys.argv[1]
units_file = 'data/units'

units = {}
with open(train_text, 'r') as fin:
    line = fin.readline()
    while line:
        line = line.strip().split(' ')
        for char in line[1:]:
            try:
                if units[char] == True:
                    continue
            except:
                units[char] = True
        line = fin.readline()

fwriter = open(units_file, 'w')
for char in units:
    print(char, file=fwriter)

In [ ]:
#!/bin/bash

#The script is to make fbank, mfcc and spectrogram from kaldi

feat_type=$1
data_dir=$2
conf_dir=conf
compress=false

if [ "$feat_type" != "fbank" || "$feat_type" != "mfcc" || "$feat_type" != "spectrogram" ]; then
    echo "Feature type $feat_type does not support!"
    exit 1;
else
    echo ============================================================================
    echo "                $feat_type Feature Extration and CMVN                          "
    echo ============================================================================

    feat_config=$conf_dir/$feat_type.conf
    if [ ! -f $feat_config ]; then
        echo "missing file $feat_config!"
        exit 1;
    fi

    x=train
    compute-$feat_type-feats --config=$feat_config scp,p:$data_dir/$x/wav_sph.scp \
                        ark,scp:$data_dir/$x/raw_$feat_type.ark,$data_dir/$x/raw_$feat_type.scp
    #compute mean and variance with all training samples
    compute-cmvn-stats --binary=false scp:$data_dir/$x/raw_$feat_type.scp $data_dir/global_${feat_type}_cmvn.txt
    #apply cmvn for training set
    apply-cmvn --norm-vars=true $data_dir/global_${feat_type}_cmvn.txt scp:$data_dir/$x/raw_$feat_type.scp ark:- |\
        copy-feats --compress=$compress ark:- ark,scp:$data_dir/$x/$feat_type.ark,$data_dir/$x/$feat_type.scp
    rm -f $data_dir/$x/raw_$feat_type.ark $data_dir/$x/raw_$feat_type.scp

	for x in dev test; do
        compute-$feat_type-feats --config=$feat_config scp,p:$data_dir/$x/wav_sph.scp ark:- | \
            apply-cmvn --norm-vars=true $data_dir/global_${feat_type}_cmvn.txt ark:- ark:- |\
                copy-feats --compress=$compress ark:- ark,scp:$data_dir/$x/$feat_type.ark,$data_dir/$x/$feat_type.scp
    done
fi

echo "Finished successfully on" `date`
exit 0

#model


In [ ]:
#!/usr/bin/python
#encoding=utf-8

import math
import torch
import torch.nn as nn
import editdistance as ed
import torch.nn.functional as F
from collections import OrderedDict

__author__ = "Ruchao Fan"

class BatchRNN(nn.Module):
    """
    Add BatchNorm before rnn to generate a batchrnn layer
    """
    def __init__(self, input_size, hidden_size, rnn_type=nn.LSTM,
                    bidirectional=False, batch_norm=True, dropout=0.1):
        super(BatchRNN, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional
        self.batch_norm = nn.BatchNorm1d(input_size) if batch_norm else None
        self.rnn = rnn_type(input_size=input_size, hidden_size=hidden_size,
                                bidirectional=bidirectional, bias=False)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        if self.batch_norm is not None:
            x = x.transpose(-1, -2)
            x = self.batch_norm(x)
            x = x.transpose(-1, -2)
        x, _ = self.rnn(x)
        x = self.dropout(x)
        #self.rnn.flatten_parameters()
        return x

class LayerCNN(nn.Module):
    """
    One CNN layer include conv2d, batchnorm, activation and maxpooling
    """
    def __init__(self, in_channel, out_channel, kernel_size, stride, padding, pooling_size=None,
                        activation_function=nn.ReLU, batch_norm=True, dropout=0.1):
        super(LayerCNN, self).__init__()
        if len(kernel_size) == 2:
            self.conv = nn.Conv2d(in_channel, out_channel, kernel_size=kernel_size, stride=stride, padding=padding)
            self.batch_norm = nn.BatchNorm2d(out_channel) if batch_norm else None
        else:
            self.conv = nn.Conv1d(in_channel, out_channel, kernel_size=kernel_size, stride=stride, padding=padding)
            self.batch_norm = nn.BatchNorm1d(out_channel) if batch_norm else None
        self.activation = activation_function(inplace=True)
        if pooling_size is not None and len(kernel_size) == 2:
            self.pooling = nn.MaxPool2d(pooling_size)
        elif len(kernel_size) == 1:
            self.pooling = nn.MaxPool1d(pooling_size)
        else:
            self.pooling = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = self.conv(x)
        if self.batch_norm is not None:
            x = self.batch_norm(x)
        x = self.activation(x)
        if self.pooling is not None:
            x = self.pooling(x)
        x = self.dropout(x)
        return x

class CTC_Model(nn.Module):
    def __init__(self, add_cnn=False, cnn_param=None, rnn_param=None, num_class=39, drop_out=0.1):
        """
        add_cnn   [bool]:  whether add cnn in the model
        cnn_param [dict]:  cnn parameters, only support Conv2d i.e.
            cnn_param = {"layer":[[(in_channel, out_channel), (kernel_size), (stride), (padding), (pooling_size)],...],
                            "batch_norm":True, "activate_function":nn.ReLU}
        rnn_param [dict]:  rnn parameters i.e.
            rnn_param = {"rnn_input_size":201, "rnn_hidden_size":256, ....}
        num_class  [int]:  the number of modelling units, add blank to be the number of classes
        drop_out [float]:  drop_out rate for all
        """
        super(CTC_Model, self).__init__()
        self.add_cnn = add_cnn
        self.cnn_param = cnn_param
        if rnn_param is None or type(rnn_param) != dict:
            raise ValueError("rnn_param need to be a dict to contain all params of rnn!")
        self.rnn_param = rnn_param
        self.num_class = num_class
        self.num_directions = 2 if rnn_param["bidirectional"] else 1
        self.drop_out = drop_out

        if add_cnn:
            cnns = []
            activation = cnn_param["activate_function"]
            batch_norm = cnn_param["batch_norm"]
            rnn_input_size = rnn_param["rnn_input_size"]
            cnn_layers = cnn_param["layer"]
            for n in range(len(cnn_layers)):
                in_channel = cnn_layers[n][0][0]
                out_channel = cnn_layers[n][0][1]
                kernel_size = cnn_layers[n][1]
                stride = cnn_layers[n][2]
                padding = cnn_layers[n][3]
                pooling_size = cnn_layers[n][4]

                cnn = LayerCNN(in_channel, out_channel, kernel_size, stride, padding, pooling_size,
                                activation_function=activation, batch_norm=batch_norm, dropout=drop_out)
                cnns.append(('%d' % n, cnn))

                try:
                    rnn_input_size = int(math.floor((rnn_input_size+2*padding[1]-kernel_size[1])/stride[1])+1)
                except:
                    #if using 1-d Conv
                    rnn_input_size = rnn_input_size
            self.conv = nn.Sequential(OrderedDict(cnns))
            rnn_input_size *= out_channel
        else:
            rnn_input_size = rnn_param["rnn_input_size"]

        rnns = []
        rnn_hidden_size = rnn_param["rnn_hidden_size"]
        rnn_type = rnn_param["rnn_type"]
        rnn_layers = rnn_param["rnn_layers"]
        bidirectional = rnn_param["bidirectional"]
        batch_norm = rnn_param["batch_norm"]
        rnn = BatchRNN(input_size=rnn_input_size, hidden_size=rnn_hidden_size, rnn_type=rnn_type,
                            bidirectional=bidirectional, dropout=drop_out, batch_norm=False)
        rnns.append(('0', rnn))
        for i in range(rnn_layers-1):
            rnn = BatchRNN(input_size=self.num_directions*rnn_hidden_size, hidden_size=rnn_hidden_size, rnn_type=rnn_type,
                                bidirectional=bidirectional, dropout=drop_out, batch_norm=batch_norm)
            rnns.append(('%d' % (i+1), rnn))
        self.rnns = nn.Sequential(OrderedDict(rnns))

        if batch_norm:
            self.fc = nn.Sequential(nn.BatchNorm1d(self.num_directions*rnn_hidden_size),
                                nn.Linear(self.num_directions*rnn_hidden_size, num_class, bias=False),)
        else:
            self.fc = nn.Linear(self.num_directions*rnn_hidden_size, num_class, bias=False)
        self.log_softmax = nn.LogSoftmax(dim=-1)

    def forward(self, x, visualize=False):
        #x: batch_size * 1 * max_seq_length * feat_size
        if visualize:
            visual = [x]

        if self.add_cnn:
            x = self.conv(x.unsqueeze(1))

            if visualize:
                visual.append(x)

            x = x.transpose(1, 2).contiguous()
            sizes = x.size()
            if len(sizes) > 3:
                x = x.view(sizes[0], sizes[1], sizes[2]*sizes[3])

            x = x.transpose(0,1).contiguous()

            if visualize:
                visual.append(x)

            x = self.rnns(x)
            seq_len, batch, _ = x.size()
            x = x.view(seq_len*batch, -1)
            x = self.fc(x)
            x = x.view(seq_len, batch, -1)
            out = self.log_softmax(x)

            if visualize:
                visual.append(out)
                return out, visual
            return out
        else:
            x = x.transpose(0, 1)
            x = self.rnns(x)
            seq_len, batch, _ = x.size()
            x = x.view(seq_len*batch, -1)
            x = self.fc(x)
            x = x.view(seq_len, batch, -1)
            out = self.log_softmax(x)
            if visualize:
                visual.append(out)
                return out, visual
            return out

    def compute_wer(self, index, input_sizes, targets, target_sizes):
        batch_errs = 0
        batch_tokens = 0
        for i in range(len(index)):
            label = targets[i][:target_sizes[i]]
            pred = []
            for j in range(len(index[i][:input_sizes[i]])):
                if index[i][j] == 0:
                    continue
                if j == 0:
                    pred.append(index[i][j])
                if j > 0 and index[i][j] != index[i][j-1]:
                    pred.append(index[i][j])
            batch_errs += ed.eval(label, pred)
            batch_tokens += len(label)
        return batch_errs, batch_tokens

    def add_weights_noise(self):
        for param in self.parameters():
            weight_noise = param.data.new(param.size()).normal_(0, 0.075).type_as(param.type())
            param = torch.nn.parameter.Parameter(param.data + weight_noise)

    @staticmethod
    def save_package(model, optimizer=None, decoder=None, epoch=None, loss_results=None, dev_loss_results=None, dev_cer_results=None):
        package = {
                'rnn_param': model.rnn_param,
                'add_cnn': model.add_cnn,
                'cnn_param': model.cnn_param,
                'num_class': model.num_class,
                '_drop_out': model.drop_out,
                'state_dict': model.state_dict()
                }
        if optimizer is not None:
            package['optim_dict'] = optimizer.state_dict()
        if decoder is not None:
            package['decoder'] = decoder
        if epoch is not None:
            package['epoch'] = epoch
        if loss_results is not None:
            package['loss_results'] = loss_results
            package['dev_loss_results'] = dev_loss_results
            package['dev_cer_results'] = dev_cer_results
        return package

if __name__ == '__main__':
    model = CTC_Model(add_cnn=True, cnn_param={"batch_norm":True, "activativate_function":nn.ReLU, "layer":[[(1,32), (3,41), (1,2), (0,0), None],
                            [(32,32), (3,21), (2,2), (0,0), None]]}, num_class=48, drop_out=0)
    for idx, m in CTC_Model.modules():
        print(idx, m)


# test_code
